# Day 1 — Hands-On Lab: PySpark & Spark SQL with Databricks Volumes

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Datasets** | ex_customers.csv (336 rows) · ex_orders.csv (4,909 rows) · ex_products.csv (1,778 rows) |
| **Storage** | Databricks Unity Catalog Volume |
| **Duration** | 60 minutes |
| **Layer** | Bronze (Delta format) |

### Learning Objectives
- Upload files into a Databricks Volume and read them from a notebook — no storage key required
- Explore multiple DataFrames with PySpark (`printSchema`, `show`, `count`)
- Join and query data with Spark SQL (temp views, `LEFT JOIN`, `YEAR`, `GROUP BY`)
- Apply PySpark transformations (`filter`, `withColumn`, `when/otherwise`)
- Identify a real data quality gap and write cleaned data to Bronze Delta tables

---
**Instructions:** Run each cell with **Shift + Enter**. Fill in any `# YOUR CODE HERE` blanks before running.

## Setup: Read From Your Volume

Replace the placeholder below with your own catalog/schema:
- `YOUR_CATALOG` → the catalog your instructor gave you access to (commonly `workspace`)
- `YOUR_SCHEMA` → the schema you created (or were given), e.g. your own name

> No key, no `spark.conf.set()`, no storage account. Unity Catalog already knows who you are and what you're allowed to read — a Volume is governed the same way as any table you'll build later in this course.

In [ ]:
# ─── Volume Setup ───────────────────────────────────────────────────────────
# HOW TO GET THIS VALUE: Catalog (left sidebar) → your catalog → your schema
# → raw_data volume → the path is shown at the top of the Volume browser.
volume_path = "/Volumes/YOUR_CATALOG/YOUR_SCHEMA/raw_data"   # ← replace with your path

customers_path = f"{volume_path}/customers/ex_customers.csv"
orders_path    = f"{volume_path}/orders/ex_orders.csv"
products_path  = f"{volume_path}/products/ex_products.csv"

# List what's actually in the volume -- a quick sanity check before reading
display(dbutils.fs.ls(volume_path))

---
## Phase A — Understand the Data

**Goal:** Read all three CSVs from the Volume and understand their structure.

### A1 — Read the Three CSVs

Use `spark.read.csv()` to load each file. We pass two options:
- `header=true` → first row is column names
- `inferSchema=true` → Spark auto-detects data types

In [ ]:
# Read all three CSVs from the Volume
customers_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(customers_path)
)

orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(orders_path)
)

products_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(products_path)
)

print(f"customers: {customers_df.count()} rows")
print(f"orders:    {orders_df.count()} rows")
print(f"products:  {products_df.count()} rows")

### A2 — Explore the Schemas

Print the schema (column names + data types) for each DataFrame.

In [ ]:
for name, df in [("customers", customers_df), ("orders", orders_df), ("products", products_df)]:
    print(f"--- {name} ---")
    df.printSchema()

customers_df.show(5, truncate=False)

**Q: How many rows are in each of the three DataFrames? Name one column Spark inferred as a date/timestamp type, and one it inferred as a string.**

*Your answer:* _______________

---
## Phase B — Spark SQL

**Goal:** Register all three DataFrames as SQL views and query — and join — them using standard SQL.

In [ ]:
customers_df.createOrReplaceTempView("customers")
orders_df.createOrReplaceTempView("orders")
products_df.createOrReplaceTempView("products")

print("All three temp views registered. You can now query them using SQL.")

### B1 — Customer Segments

How many customers fall into each segment? Sort by count descending.

In [ ]:
result = spark.sql("""
    SELECT segment,
           COUNT(*) AS customer_count
    FROM customers
    GROUP BY segment
    ORDER BY customer_count DESC
""")
result.show()

**Q: Which customer segment has the most customers? How many?**

*Your answer:* _______________

### B2 — Orders by Status

Count orders by `order_status`, sorted from most common to least common.

In [ ]:
result = spark.sql("""
    SELECT order_status,
           COUNT(*) AS order_count
    FROM orders
    GROUP BY order_status
    ORDER BY order_count DESC
""")
result.show()

**Q: Which order status is the most common? Roughly what percentage of all orders does it represent?**

*Your answer:* _______________

### B3 — Customers Who Have Never Ordered

GlobalMart's marketing team wants to send a "we miss you" email to any customer who has never placed a single order. Use a `LEFT JOIN` between `customers` and `orders`, keeping only rows where no matching order exists.

In [ ]:
result = spark.sql("""
    SELECT c.customer_id, c.customer_name, c.segment
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.order_id IS NULL
""")
result.show()
print(f"Customers with zero orders: {result.count()}")

**Q: How many customers have never placed an order? List their `customer_id` values.**

*Your answer:* _______________

### B4 — Orders Per Year

Count orders placed in each year, using `YEAR(order_purchase_date)`. Sort from most recent year to oldest.

In [ ]:
result = spark.sql("""
    SELECT YEAR(order_purchase_date) AS order_year,
           COUNT(*) AS order_count
    FROM orders
    GROUP BY order_year
    ORDER BY order_year DESC
""")
result.show()

**Q: In which year were the most orders placed? How many orders were placed that year?**

*Your answer:* _______________

---
## Phase C — PySpark Transformations

**Goal:** Use PySpark's DataFrame API to filter rows, add computed columns, and fix data quality issues.

### C1 — Calculate Delivery Time

Filter to delivered orders and add a `delivery_days` column measuring the gap between purchase and delivery.

In [ ]:
from pyspark.sql.functions import col, datediff, when, trim, avg

delivered_orders = orders_df.filter(col("order_status") == "delivered")

delivered_with_days = delivered_orders.withColumn(
    "delivery_days",
    datediff(col("order_delivered_date"), col("order_purchase_date"))
)

print(f"Delivered orders: {delivered_with_days.count()}")

In [ ]:
delivered_with_days.agg(
    avg("delivery_days").alias("avg_delivery_days")
).show()

**Q: What is the average delivery time, in days, for delivered orders?**

*Your answer:* _______________

### C2 — Classify Orders by Delivery Speed

Add a `delivery_speed` column:
- **3 days or fewer** → `"Fast"`
- **4–7 days** → `"Normal"`
- **More than 7 days** → `"Slow"`

Use `when / otherwise`.

In [ ]:
orders_with_speed = delivered_with_days.withColumn(
    "delivery_speed",
    when(col("delivery_days") <= 3, "Fast")
    .when(col("delivery_days") <= 7, "Normal")
    .otherwise("Slow")
)

orders_with_speed.groupBy("delivery_speed").count().orderBy("delivery_speed").show()

**Q: How many delivered orders fall into each delivery speed tier (Fast / Normal / Slow)?**

*Your answer:* _______________

---
## Phase D — Data Quality Check

**Goal:** Real product catalogs are rarely complete. Check how much of `products` is missing a `manufacturer` value.

In [ ]:
# Find how many products are missing a manufacturer value
total_products = products_df.count()
missing_manufacturer = products_df.filter(
    col("manufacturer").isNull() | (trim(col("manufacturer")) == "")
).count()

pct_missing = round(100 * missing_manufacturer / total_products, 1)
print(f"Products missing manufacturer: {missing_manufacturer} / {total_products} ({pct_missing}%)")

products_df.groupBy("category").count().orderBy(col("count").desc()).show()

**Q: What percentage of products are missing a `manufacturer` value? Why is it risky to write this straight to Bronze without flagging the gap, even though Bronze is supposed to stay "raw and unmodified"?**

*Your answer:* _______________

---
## Phase E — Write to Bronze

**Goal:** Persist all three DataFrames as Delta tables in the Bronze layer, inside your own Volume.

In [ ]:
bronze_path = f"{volume_path}/bronze"

customers_df.write.format("delta").mode("overwrite").save(f"{bronze_path}/customers")
orders_df.write.format("delta").mode("overwrite").save(f"{bronze_path}/orders")
products_df.write.format("delta").mode("overwrite").save(f"{bronze_path}/products")

for name in ["customers", "orders", "products"]:
    written_df = spark.read.format("delta").load(f"{bronze_path}/{name}")
    print(f"bronze/{name}: {written_df.count()} rows written")

**Q: Do the Bronze row counts match the source CSVs? What are the three counts?**

*Your answer:* _______________

---
## Submission Checklist

Before uploading this notebook, fill in each blank and verify the checklist.

```
Submission Checklist
────────────────────────────────────────────────────────
✅ Volume created and three folders (customers/, orders/, products/) uploaded
✅ Notebook connected — dbutils.fs.ls() showed all three folders
── Total rows (customers / orders / products):     ______
── Top customer segment:                           ______
── Most common order status:                       ______
── Customers with zero orders:                      ______
── Year with most orders:                           ______
── Average delivery time (days):                    ______
── Fast / Normal / Slow order counts:                ______
── % of products missing manufacturer:               ______
── Bronze Delta row counts (customers/orders/products): ______
────────────────────────────────────────────────────────
```